<a href="https://colab.research.google.com/github/Richard-IA86/Data-Analytics-Workspace/blob/main/TP4/Actividad_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clase 4 — Limpieza de Datos con Pandas

**Actividades:** Identificación de valores nulos y duplicados.

## Actividad 1: Identificación de Valores Nulos y Duplicados

**Dataset:** `satis_clientes.csv`  
**Objetivo:** Cargar los datos con Pandas, identificar filas con datos faltantes y contar los valores nulos por columna.

In [ ]:
import pandas as pd
from google.colab import files
import io

# Subir el archivo satis_clientes.csv cuando aparezca el selector
print("Seleccionar: satis_clientes.csv")
uploaded = files.upload()

nombre_archivo = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[nombre_archivo]))

print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()

### 1.1 — Identificación de filas con datos faltantes (`isnull()`)

In [ ]:
# Máscara booleana: True donde hay al menos un nulo en la fila
mask_nulos = df.isnull().any(axis=1)

print(f"Filas con al menos un valor nulo: {mask_nulos.sum()}")
print()

# Mostrar las filas afectadas
df[mask_nulos]

### 1.2 — Conteo de valores nulos por columna

In [ ]:
nulos_por_columna = df.isnull().sum()
porcentaje = (df.isnull().mean() * 100).round(2)

resumen_nulos = pd.DataFrame({
    "Nulos": nulos_por_columna,
    "% del total": porcentaje,
})

print("Resumen de valores nulos por columna:")
print(resumen_nulos[resumen_nulos["Nulos"] > 0])

## Actividad 2: Exploración y limpieza preliminar

**Dataset:** `Actividad 2.xlsx - Hoja 1.csv` — temperatura corporal de personas durante 10 días  
**Objetivo:** Identificar duplicados y analizar el mejor tratamiento para datos anómalos y nulos.

### 2.1 — Identificación de filas duplicadas

In [ ]:
# Subir el archivo "Actividad 2.xlsx - Hoja 1.csv" cuando aparezca el selector
print("Seleccionar: Actividad 2.xlsx - Hoja 1.csv")
uploaded2 = files.upload()

nombre_archivo2 = list(uploaded2.keys())[0]
df2 = pd.read_csv(io.BytesIO(uploaded2[nombre_archivo2]))

print(f"Dimensiones: {df2.shape[0]} filas × {df2.shape[1]} columnas")
print()
df2.head(10)

In [ ]:
duplicados = df2[df2.duplicated(keep=False)]

print(f"Filas duplicadas encontradas: {df2.duplicated().sum()}")
print()

# Ver los duplicados ordenados por nombre para compararlos fácilmente
duplicados.sort_values("nombre")

### 2.2 — Valores nulos por columna (día)

In [ ]:
nulos_temp = df2.isnull().sum()
porcentaje_temp = (df2.isnull().mean() * 100).round(2)

resumen_temp = pd.DataFrame({
    "Nulos": nulos_temp,
    "% del total": porcentaje_temp,
})

print("Valores nulos por columna:")
print(resumen_temp[resumen_temp["Nulos"] > 0])

### 2.3 — Datos fuera de rango fisiológico (< 35.0°C o > 38.5°C)

In [ ]:
cols_dias = [c for c in df2.columns if c != "nombre"]

# Máscara: celdas fuera del rango fisiológico
fuera_rango = (df2[cols_dias] < 35.0) | (df2[cols_dias] > 38.5)

print(f"Total de registros fuera de rango: {fuera_rango.values.sum()}")
print()
print("Conteo por día:")
print(fuera_rango.sum())

### 2.4 — Análisis: tratamiento recomendado para datos anómalos y nulos

| Tipo de dato | Problema detectado | Tratamiento recomendado |
|---|---|---|
| Valores nulos (`d1`…`d10`) | Temperatura no registrada ese día | Imputar con la media de la persona (por fila) si faltan pocos días; si faltan muchos, descartar la fila |
| Filas duplicadas | Persona registrada más de una vez con los mismos valores | Eliminar duplicados con `drop_duplicates()`, conservando la primera ocurrencia |
| Datos fuera de rango (< 35°C o > 38.5°C) | Valores fisiológicamente imposibles o extremos | Reemplazar con `NaN` y tratar como nulos; no imputar sin revisar con el área médica |

> **Regla general en SynthData:** Antes de eliminar o imputar, documentar cuántos registros se ven afectados y consultar con el stakeholder si esos casos tienen explicación.